# Cross-Model Evaluation and Best-Model Comparison

Evaluation-only workflow for MobileNetV2, EfficientNetB0, EfficientNet-Lite0, MobileNetV3-Small, and ShuffleNetV2 0.5x.
Architectures: mobilenetv2, efficientnet_b0, efficientnet_lite0, mobilenetv3_small, shufflenetv2_05.
It compares float and available TFLite artifacts without retraining or overwriting model files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
REPO_DIR = '/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
DATASET_BASE = '/content/drive/MyDrive/leaf_data/processed'
MODEL_ROOT = '/content/drive/MyDrive/leaf_models'
OUTPUT_DIR = '/content/drive/MyDrive/leaf_model_comparison'
MAX_TEST_SAMPLES = None  # Set to 500 for a fast smoke evaluation.
EVAL_BATCH_SIZE = 32
print('Dataset:', DATASET_BASE)
print('Models:', MODEL_ROOT)
print('Results:', OUTPUT_DIR)

## Environment setup

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
required_paths = ['requirements.txt', 'training/train.py', 'training/evaluate.py', 'notebooks/model_comparison_common.py']
missing_paths = [path for path in required_paths if not os.path.isfile(os.path.join(REPO_DIR, path))]
if missing_paths:
    raise RuntimeError(f'Repository clone is incomplete; missing: {missing_paths}')
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'notebooks'))
print('Complete project repository is ready:', REPO_DIR)

## Install project dependencies

In [ ]:
import subprocess
import sys

requirements_path = os.path.join(REPO_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', requirements_path], check=True)
print('Dependencies installed from:', requirements_path)

## Evaluate all five architectures

In [ ]:
from model_comparison_common import run_comparison

recommendation, summary_df, ranking_df = run_comparison(
    repo_dir=REPO_DIR,
    dataset_base=DATASET_BASE,
    model_root=MODEL_ROOT,
    output_dir=OUTPUT_DIR,
    max_test_samples=MAX_TEST_SAMPLES,
    eval_batch_size=EVAL_BATCH_SIZE,
)
display(summary_df)
display(ranking_df)

## Inspect the recommendation and plots

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

print('Best float model:', recommendation['best_float_accuracy'])
print('Best edge model:', recommendation['best_edge_deployment'])
print('Best balanced model:', recommendation['best_balanced'])

for plot_name in ('accuracy_f1_comparison.png', 'size_vs_latency.png', 'float_to_int8_drop.png', 'pareto_frontier.png'):
    plot_path = Path(OUTPUT_DIR) / 'plots' / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path)))

report_path = Path(OUTPUT_DIR) / 'best_model_report.md'
if report_path.exists():
    display(Markdown(report_path.read_text(encoding='utf-8')))
print('All outputs:', OUTPUT_DIR)
print('Archive:', f'{OUTPUT_DIR}.zip')